# Script to fetch Tourplay rosters from tournament urls

In [ ]:
import numpy as np
import pandas as pd
import json
import time

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

options = Options()
#options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

In [ ]:
from PIL import Image
from IPython.display import display

## Fetch list of rosters to fetch

In [ ]:
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)


In [ ]:
# new batch for 20-21 dec
url_list = [ 
    'https://tourplay.net/en/blood-bowl/tellurian-christmas-brawl-2025/',
    'https://tourplay.net/en/blood-bowl/simon-whitehouse/',
    'https://tourplay.net/en/blood-bowl/quickbowl-season-kickoff/',
    'https://tourplay.net/en/blood-bowl/set-de-cawa-2025/',
    'https://tourplay.net/en/blood-bowl/barrow-lords-christmas-bash/',
    'https://tourplay.net/en/blood-bowl/yorkshiremas-bowl-ii/',
    'https://tourplay.net/en/blood-bowl/torneo-navideno-morada-del-dragon-25/'

]
    #'https://tourplay.net/en/blood-bowl/torneo-navidad-blood-bowl-2025/'  # 13/12 # 7 # GIVES A SELENIUM ERROR!! overlapping element caused by long coach name
    # 'https://tourplay.net/en/blood-bowl/mftme-december/' , # 13/12' , # 17
    # 'https://tourplay.net/en/blood-bowl/hooded-goblin-christmas-tournament/' , # 13/12 # 15
    # 'https://tourplay.net/en/blood-bowl/the-prairie-oyster/' , # 13/12 24
    # 'https://tourplay.net/en/blood-bowl/christmas-carnage/' # 13/12 15
    # 'https://tourplay.net/en/blood-bowl/welcome-to-the-3rd-edition/', # 16
    # 'https://tourplay.net/en/blood-bowl/third-edition-tryout-tournament/', # 6
    # 'https://tourplay.net/en/blood-bowl/berkshire-blitz-pre-season-brawl/', # 10
    # 'https://tourplay.net/en/blood-bowl/december-big-brain-bloodbath/', # 8
    # 'https://tourplay.net/en/blood-bowl/zhufbar-winter-cup-vi/', # 20
    # 'https://tourplay.net/en/blood-bowl/soulforge-bb2025-starters-tournament/', # 6
    # 'https://tourplay.net/en/blood-bowl/wtkm-v-kibice-kontratakuja/', # 13/12 # 16
    # 'https://tourplay.net/en/blood-bowl/red-bug-cup/', # 13/12 # 12'


In [ ]:
# Note maximize Chrome browser before running this part

ids = []
player_names = []
urls = []

for url in url_list:
    print(url)
    driver.get(url + 'classifications')
    time.sleep(3)
    elements = driver.find_elements(By.CLASS_NAME, 'user-name-to-show')
    cnt = len(elements)
    print(str(cnt) + " rosters available at url")

    for counter in range(cnt):
        ids.append(counter)
        elements = driver.find_elements(By.CLASS_NAME, 'user-name-to-show')
        el = elements[counter]
        player_names.append(el.text)
        el.click()
        time.sleep(2)
        print(driver.current_url)
        urls.append(driver.current_url)
        driver.back()
        time.sleep(2)

# quit Chrome Browser session
driver.quit()


In [ ]:
# create list of tuples
data = zip(ids, player_names, urls)
# create dataframe from list
df_rosters_to_fetch = pd.DataFrame(data, columns=['row_id', 'coach_name', 'roster_url'])
df_rosters_to_fetch['roster_id'] = df_rosters_to_fetch['roster_url'].transform(lambda x: x.split("/")[-1])
df_rosters_to_fetch.to_csv('rosters_to_fetch_C.csv')

## Fetch rosters

In [ ]:
df_rosters_to_fetch = pd.concat([pd.read_csv('rosters_to_fetch_A.csv'), 
                                pd.read_csv('rosters_to_fetch_B.csv'),
                                pd.read_csv('rosters_to_fetch_C.csv'),
                                ], ignore_index=True)

In [ ]:
df_rosters_to_fetch.shape

In [ ]:
%run src/get_team_roster.py

# first roster id to fetch
roster_id = df_rosters_to_fetch.iloc[0]['roster_id']

df_roster, roster = get_tourplay_roster(roster_id)

In [ ]:
df_roster


In [ ]:
%run src/get_team_roster.py

cnt = 0
for roster_id in df_rosters_to_fetch['roster_id']:
    if cnt == 0:
        df_roster, roster = get_tourplay_roster(roster_id)
        df_rosters = df_roster
    else:
        df_roster, roster = get_tourplay_roster(roster_id)
        df_rosters = pd.concat([df_rosters, df_roster], ignore_index=True)
    cnt = cnt + 1
    print(".", end = '')

In [ ]:
target = 'datasets/current/df_rosters_third_season_v2'

df_rosters.to_csv(target + '.csv')

In [ ]:
df_rosters.shape